# 01: comprehensive FX data pull (run on a Bloomberg terminal)

A topic-free pull of the full FX dataset: everything FX, not scoped to any one paper or strategy. It is the only price pull in the repo, having absorbed the narrower spot, forward and Treasury bill pull that used to sit beside it. `02_bloomberg_pull_macro.ipynb` is the companion for country macro series.

Run it on a machine with a live Terminal session. It writes seven long-format parquets into `data/raw/` and leaves the DVC step to run once at the bottom of the notebook. `notebooks/00_data_pipeline.ipynb` covers that DVC and Box hop, and what to check after.

### Binds with what is already in `data/raw`

Every dataset here uses the same long format as the rest of the pull, `ticker, date, field, value`, with `PX_LAST / PX_BID / PX_ASK`. Reading is by ticker selection rather than by file, so a narrower piece of work passes a smaller label map and gets a narrower panel out of the same file: `ParquetSource.quotes` takes whatever `Catalog.label_map` hands it. Adding currencies to a pull therefore does not disturb anything already reading it.

### The Bloomberg limits, and how this respects them

* **Monthly, about 5000 to 7000 unique securities.** This pull is a few thousand unique securities, well under the cap. A security reused in the same month is free, so batching across a day costs nothing. It is a shared terminal, so still coordinate if teammates are pulling heavily.
* **Daily, 500,000 hits (one request for one security and field).** A full refresh is a few thousand pairings, far under the cap. Pull once, do not refresh again, and keep to end of day (`Per="D"`), which is cheaper than intraday.
* **Open real-time fields, 3500.** `blp.bdh` returns static history rather than live fields, so scripted pulls barely touch this. Batching in chunks keeps it safe.

If you ever see `#N/A Limit`, a cap was hit. Split the batch and resume the next day for the daily cap, or wait for the month reset for the monthly one.

The ticker counts this notebook prints come from `Catalog`, so they move when the currency universe in `fxcarry.reference` changes. Check them against `notebooks/data_dictionary/` before trusting a fresh pull.

> **This notebook needs a live Bloomberg Terminal.** `xbbg` talks to a local BLPAPI
> session, so nothing below runs on a machine without one, and the cells are stored
> without output for that reason.
>
> The code is current with the class API: ticker construction comes from `Catalog` and
> `Currency` (`fwd_ticker`, `vol_ticker`), the reference tables from `fxcarry.reference`,
> and the read-back at the end from `ParquetSource`. File names and the pull window live
> in this notebook rather than in the library, which holds market conventions only.
>
> It was last executed against the previous function-based library. The pulls it
> produced are what `data/raw` currently holds, tracked by DVC, and
> `notebooks/data_dictionary/` documents them. Re-run this only when you actually want
> to refresh the data, and check the printed ticker counts against that documentation
> first: the universe has grown since the last run, so the counts will not match the
> old ones.

In [ ]:
from pathlib import Path

import pandas as pd
from xbbg import blp

from fxcarry import Catalog, reference

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# The library holds market conventions, not file names or pull windows, so those sit
# here beside the pull that uses them rather than in `reference`.
START = "1983-11-01"
END = pd.Timestamp.today().strftime("%Y-%m-%d")     # pull through today
TBILL_TICKER = "GB1M Index"
FILES = {
    "spot": "spot_daily.parquet",
    "fwd_1m": "fwd_points_1m_daily.parquet",
    "fwd_curve": "fwd_points_multi_daily.parquet",
    "vol": "fx_vol_daily.parquet",
    "index": "fx_dollar_index_daily.parquet",
    "rates": "fx_short_rate_daily.parquet",
    "tbill": "tbill_daily.parquet",
}

# Every currency with a market pair, including the ones the euro replaced in 1999.
CATALOG = Catalog.with_legacy()
UNIVERSE = {**reference.SPOT_FWD_TICKERS, **reference.LEGACY_EURO_TICKERS}
VOL_CATALOG = CATALOG.subset([c for c in reference.VOL_CURRENCIES if c in CATALOG])

print(f"Window:   {START} -> {END}")
print(f"Universe: {len(CATALOG)} currencies")
print(f"Vol pairs: {len(VOL_CATALOG)}  ATM tenors={reference.VOL_TENORS}  "
      f"wings={reference.VOL_DELTAS}d")
print(f"Rates:     {len(reference.SHORT_RATE_TICKERS)} ccy, "
      f"{len(CATALOG.tickers('rate'))} tickers, tenors {reference.RATE_TENORS}")

## Step 0. Validate the tickers cheaply first

Pull a handful of representative tickers before the big batches, to confirm the ticker formats resolve on your terminal. The option and basis tickers are the ones to check: the vol source suffix (`BGN`) and the basis codes vary by terminal.

In [ ]:
eur = CATALOG["EUR"]
sample = [
    eur.spot_ticker,                             # spot EURUSD
    eur.fwd_ticker("3M"),                        # 3M forward points
    eur.vol_ticker("atm", "1M"),                 # ATM 1M vol
    eur.vol_ticker("rr", "1M", 25),              # 25 delta risk reversal
    eur.fwd_ticker("2Y"),                        # long-end forward point
    eur.vol_ticker("rr", "1M", 5),               # 5 delta deep-tail wing
    reference.SHORT_RATE_TICKERS["USD"]["3M"],   # G10 short rate
    reference.SHORT_RATE_TICKERS["MXN"]["1M"],   # EM short rate (verify)
]
check = blp.bdh(tickers=sample, flds=[reference.PX_LAST], start_date="2024-01-01",
                end_date=END, Per="D", backend="pandas")
print("resolved tickers:")
print(check["ticker"].unique() if "ticker" in check.columns else check.columns.tolist())
check.tail()

Validate the EM (and non-LIBOR G10) rate tickers

In [ ]:
import pandas as pd
def probe_rate(t):
    try:
        df = blp.bdh(tickers=t, flds=["PX_LAST"], start_date="1990-01-01",
                     end_date=END, Per="D", backend="pandas")
        v = df["value"].dropna() if "value" in df.columns else pd.Series(dtype=float)
        if len(v):
            start = str(df["date"].min())[:7] if "date" in df.columns else "?"
            return ("OK   ", len(v), start, round(float(v.iloc[-1]), 3))
        return ("EMPTY", 0, "-", None)
    except Exception as e:
        return ("FAIL ", 0, "-", str(e).splitlines()[-1][:28])

for ccy, curve in reference.SHORT_RATE_TICKERS.items():
    for tenor, tk in curve.items():
        status, n, start, last = probe_rate(tk)
        print(f"{ccy} {tenor:3s} {tk:16s} {status} rows={n:5d} start={start} last={last}")

In [ ]:
for t in ["MIFOR3M Index", "MIFOR C Curncy", "IUDTB91 Index", "GIND3M Index",
          "IRSWO3 Curncy", "IN0003M Index", "NSEMIBOR Index"]:
    print(f"{t:16s}", probe_rate(t))

In [ ]:
def bdh_batched(tickers, flds, start=START, end=END, batch=150):
    """Pull a long list in chunks (keeps each request modest and lets you
    resume if a batch trips a limit). Returns one concatenated long frame."""
    tickers = list(dict.fromkeys(tickers))     # dedupe, keep order
    frames = []
    for i in range(0, len(tickers), batch):
        chunk = tickers[i : i + batch]
        frames.append(blp.bdh(tickers=chunk, flds=flds, start_date=start,
                              end_date=end, Per="D", backend="pandas"))
        print(f"  {i + len(chunk):4d}/{len(tickers)} pulled")
    return pd.concat(frames, ignore_index=True)

## Dataset 1. Spot, full universe (extends `spot_daily.parquet`)

In [ ]:
spot_tickers = [spot for spot, _ in UNIVERSE.values()]
spot = bdh_batched(spot_tickers, reference.FIELDS)
spot.to_parquet(DATA_DIR / FILES["spot"])
print(f"Spot: {spot.shape} -> {FILES['spot']}")

## Dataset 2. 1M forward points, full universe (extends `fwd_points_1m_daily.parquet`)

In [ ]:
fwd1m_tickers = [fwd for _, fwd in UNIVERSE.values()]
fwd1m = bdh_batched(fwd1m_tickers, reference.FIELDS)
fwd1m.to_parquet(DATA_DIR / FILES["fwd_1m"])
print(f"Fwd 1M: {fwd1m.shape} -> {FILES['fwd_1m']}")

## Dataset 3. The forward curve, all tenors (`fwd_points_multi_daily.parquet`)

In [ ]:
# Catalog.tickers builds every forward-points ticker across the catalog, using each
# currency's own forward root, so the NDF names come out right without a special case.
fwd_curve_tickers = CATALOG.tickers("forward", reference.FWD_TENORS)
fwd_curve = bdh_batched(fwd_curve_tickers, reference.FIELDS)
fwd_curve.to_parquet(DATA_DIR / FILES["fwd_curve"])
print(f"Fwd curve: {fwd_curve.shape} -> {FILES['fwd_curve']}")

## Dataset 4. FX option volatility surfaces (`fx_vol_daily.parquet`)

The full surface for the liquid pairs: the ATM term structure plus the 10 and 25 delta risk reversal and butterfly wings at every tenor. Mid, bid, and ask (`PX_LAST / PX_BID / PX_ASK`), so option spreads are captured now for cost modelling of any hedged or vol-carry strategy. Bid/ask can be sparse for some `BGN` wings; whatever populates is kept, and mid is always there.

In [ ]:
# The at-the-money term structure, plus a risk reversal and a butterfly at every
# tenor and wing delta. Restricted to the currencies that quote a surface at all.
vol_tickers = (VOL_CATALOG.tickers("atm", reference.VOL_TENORS)
               + VOL_CATALOG.tickers("rr", reference.SMILE_TENORS, reference.VOL_DELTAS)
               + VOL_CATALOG.tickers("bf", reference.SMILE_TENORS, reference.VOL_DELTAS))
print(f"{len(vol_tickers)} vol tickers")
vol = bdh_batched(vol_tickers, reference.FIELDS)
vol.to_parquet(DATA_DIR / FILES["vol"])
print(f"Vol surface: {vol.shape} -> {FILES['vol']}")

## Dataset 5. Dollar indices (`fx_dollar_index_daily.parquet`)

In [ ]:
idx_tickers = list(reference.DOLLAR_INDEX_TICKERS.values())
idx = bdh_batched(idx_tickers, reference.FIELDS)
idx.to_parquet(DATA_DIR / FILES["index"])
print(f"Dollar indices: {idx.shape} -> {FILES['index']}")

## Dataset 6. Short-term interest rates (`fx_short_rate_daily.parquet`)

One short rate per currency for the direct, rate-differential definition of carry (the deposit/OIS differential), a complement to the forward-implied carry from spot and forwards. With the forwards these also pin the CIP deviation, i.e. the cross currency basis. G10 uses the classic 3M IBOR fixings (decades of history, matching BER's window); EM uses local money-market or policy proxies and is the least certain part of the pull, so read the Step 0 samples and prune what does not resolve.

In [ ]:
rate_tickers = CATALOG.tickers("rate")
rates = bdh_batched(rate_tickers, [reference.PX_LAST])
rates.to_parquet(DATA_DIR / FILES["rates"])
print(f"Short rates: {rates.shape} -> {FILES['rates']}")

## Refresh the Treasury bill through today (keeps `tbill_daily.parquet` current)

In [ ]:
tbill = blp.bdh(tickers=TBILL_TICKER, flds=[reference.PX_LAST],
                start_date=START, end_date=END, Per="D", backend="pandas")
tbill.to_parquet(DATA_DIR / FILES["tbill"])
print(f"T-bill: {tbill.shape} -> {FILES['tbill']}")

## Track everything with DVC (run once, in a shell)

Start the rclone bridge first, then add every parquet (the three existing ones were pulled again, so their content and pointers update too), commit the pointers, and push to Box.

```bash
rclone serve webdav uchicago-box:fxcarry-data --addr 127.0.0.1:8080 --vfs-cache-mode writes

cd fxcarry
dvc add data/raw/spot_daily.parquet data/raw/fx_short_rate_daily.parquet data/raw/fwd_points_1m_daily.parquet data/raw/fwd_points_multi_daily.parquet data/raw/fx_vol_daily.parquet data/raw/fx_dollar_index_daily.parquet data/raw/tbill_daily.parquet
git add data/raw/*.dvc
git commit -m "data: comprehensive FX pull (full universe, forward curve, vol surfaces, basis, indices)"
dvc push
```

Teammates and the other machine then just `git pull` and `dvc pull`.

## Validation checkpoint

Spot and forwards read back through the one path the library has: a `ParquetSource` over the
file, a label map off the catalog saying which ticker becomes which column, and a pivot into
`Quotes`. The same call covers the full universe simply by passing a wider map. The vol,
forward-curve, rate and index files are long frames in the same shape, and
`notebooks/data_dictionary/` reads each of them. A quick coverage read on each:

In [ ]:
from fxcarry import ParquetSource

# One read path for every pull: a source, a label map off the catalog, then pivot.
spot_quotes = ParquetSource(DATA_DIR / FILES["spot"]).quotes(CATALOG.label_map("spot"))
print("Spot coverage (full universe):")
display(spot_quotes.coverage())

for name, key in [("fwd curve", "fwd_curve"), ("vol", "vol"),
                  ("indices", "index"), ("rates", "rates")]:
    df = pd.read_parquet(DATA_DIR / FILES[key])
    n_ids = df["ticker"].nunique() if "ticker" in df.columns else df.shape[1]
    span = (df["date"].min(), df["date"].max()) if "date" in df.columns else ("?", "?")
    print(f"{name:10s}: {df.shape}  ids={n_ids}  span={span}")